**Prerequisite to run this notebook:** having [almond jupyter plugin](https://almond.sh/) installed, and running cells using a Scala kernel

To use Spark, we can use `$ivy` magic: https://almond.sh/docs/usage-spark

In [13]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

We can also use `ivy` to load project source code from local Ivy/Maven cache (run `make publish-local` to update):

In [14]:
val version = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % version)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

version: String = "0.0.1"

Import libraries (Spark, dataprovenance project, ...):

In [15]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan

import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._

import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.execution.SparkPlan
import org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations._

Initialize Spark session:

In [16]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook")
  .getOrCreate()

spark: SparkSession = org.apache.spark.sql.classic.SparkSession@d147928

Create Spark dataframe:

In [17]:
val df: DataFrame = spark.createDataFrame(
    Seq(
        ("A", Date.valueOf("2026-01-15"), 10.0, 90),
        ("A", Date.valueOf("2026-01-16"), 10.0, 120),
        ("A", Date.valueOf("2026-01-17"), 5.0, 300),
        ("B", Date.valueOf("2026-01-15"), 100.0, 20),
        ("B", Date.valueOf("2026-01-16"), 100.0, 30),
        ("B", Date.valueOf("2026-01-17"), 80.0, 60)
    )
).toDF("product", "date", "price", "sales")

df.show()

+-------+----------+-----+-----+
|product|      date|price|sales|
+-------+----------+-----+-----+
|      A|2026-01-15| 10.0|   90|
|      A|2026-01-16| 10.0|  120|
|      A|2026-01-17|  5.0|  300|
|      B|2026-01-15|100.0|   20|
|      B|2026-01-16|100.0|   30|
|      B|2026-01-17| 80.0|   60|
+-------+----------+-----+-----+



df: DataFrame = [product: string, date: date ... 2 more fields]

Apply provenance:
(Note that because we imported implicit class `org.dataprov.dp.sparkdataprovenance.DataFrameProvenanceTransformations.DataFrameWithProvenance`, all Spark `DataFrame` are automatically *"monkey patched"* to `DataFrameWithProvenance`, see [Implicit Class in Scala](https://docs.scala-lang.org/overviews/core/implicit-classes.html) )

In [18]:
df.addProvenanceColumn.show()

+-------+----------+-----+-----+---------------+
|product|      date|price|sales|_provenance_tag|
+-------+----------+-----+-----+---------------+
|      A|2026-01-15| 10.0|   90|              0|
|      A|2026-01-16| 10.0|  120|              1|
|      A|2026-01-17|  5.0|  300|              2|
|      B|2026-01-15|100.0|   20|              3|
|      B|2026-01-16|100.0|   30|              4|
|      B|2026-01-17| 80.0|   60|              5|
+-------+----------+-----+-----+---------------+



Get logical + physical plans:

In [22]:
val df2 = df.groupBy("product")
  .agg(
    sum("sales").as("total_sales"),
    avg("price").as("avg_price")
  )
df2.show()

df.explain(true)  // print logical plans

// 1. Parsed Logical Plan (Unresolved)
val parsedPlan: LogicalPlan = df.queryExecution.logical

// 2. Analyzed Logical Plan (Resolved)
// This is usually what you want when inspecting the schema and resolved columns
val analyzedPlan: LogicalPlan = df.queryExecution.analyzed

// 3. Optimized Logical Plan
// This is the plan after Spark's Catalyst optimizer has applied its rules
val optimizedPlan: LogicalPlan = df.queryExecution.optimizedPlan

// 4. Physical Plan
// This is the plan produced by the physical planning strategies, 
// but BEFORE rule-based physical optimizations (like whole-stage codegen) are applied.
val physicalPlan: SparkPlan = df.queryExecution.sparkPlan

// 5. Executed Physical Plan (Final)
// This is the absolute final plan that Spark actually runs. It includes 
// execution-specific preparations like adding Exchange nodes (shuffles) 
// and WholeStageCodegen blocks.
val executedPlan: SparkPlan = df.queryExecution.executedPlan

26/05/01 00:28:25 INFO DAGScheduler: Registering RDD 50 ($anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768) as input to shuffle 8
26/05/01 00:28:25 INFO DAGScheduler: Got map stage job 16 ($anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768) with 6 output partitions
26/05/01 00:28:25 INFO DAGScheduler: Final stage: ShuffleMapStage 24 ($anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768)
26/05/01 00:28:25 INFO DAGScheduler: Parents of final stage: List()
26/05/01 00:28:25 INFO DAGScheduler: Missing parents: List()
26/05/01 00:28:25 INFO DAGScheduler: Missing parents found for ShuffleMapStage 24: List()
26/05/01 00:28:25 INFO DAGScheduler: Submitting ShuffleMapStage 24 (MapPartitionsRDD[50] at $anonfun$withThreadLocalCaptured$2 at CompletableFuture.java:1768), which has no missing parents
26/05/01 00:28:25 INFO MemoryStore: Block broadcast_16 stored as values in memory (estimated size 35.5 KiB, free 2.1 GiB)
26/05/01 00:28:25 INFO MemoryStore: 

+-------+-----------+-----------------+
|product|total_sales|        avg_price|
+-------+-----------+-----------------+
|      A|        510|8.333333333333334|
|      B|        110|93.33333333333333|
+-------+-----------+-----------------+

== Parsed Logical Plan ==
'UnresolvedSubqueryColumnAliases [product, date, price, sales]
+- LocalRelation [_1#173, _2#174, _3#175, _4#176]

== Analyzed Logical Plan ==
product: string, date: date, price: double, sales: int
Project [_1#173 AS product#177, _2#174 AS date#178, _3#175 AS price#179, _4#176 AS sales#180]
+- LocalRelation [_1#173, _2#174, _3#175, _4#176]

== Optimized Logical Plan ==
LocalRelation [product#177, date#178, price#179, sales#180]

== Physical Plan ==
LocalTableScan [product#177, date#178, price#179, sales#180]



df2: DataFrame = [product: string, total_sales: bigint ... 1 more field]
parsedPlan: LogicalPlan = UnresolvedSubqueryColumnAliases(
  outputColumnNames = ArraySeq("product", "date", "price", "sales"),
  child = LocalRelation(
    output = List(
      AttributeReference(
        name = "_1",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      AttributeReference(
        name = "_2",
        dataType = DateType,
        nullable = true,
        metadata = {}
      ),
      AttributeReference(
        name = "_3",
        dataType = DoubleType,
        nullable = false,
        metadata = {}
      ),
      AttributeReference(
        name = "_4",
        dataType = IntegerType,
        nullable = false,
        metadata = {}
      )
    ),
    data = List(
      [A,20468,10.0,90],
      [A,20469,10.0,120],
      [A,20470,5.0,300],
      [B,20468,100.0,20],
      [B,20469,100.0,30],
      [B,20470,80.0,60]
    ),
    isStreaming = false,
    stream 